In [0]:
import sys
import requests
import json
import time
from pyspark.sql import functions as F

#Load Socrata app token
sys.path.append("/Workspace/Users/pranayghadigaonkar1912@gmail.com/config")
from app_secrets import SOCRATA_APP_TOKEN

#Socrata API config
SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

#Date range - Q1  2025 (Jan - Mar)
CREATED_DATE_FROM = "2025-01-01"
CREATED_DATE_TO = "2025-02-01" #Exclusive

#Volumn Path for raw JSON
VOLUME_PATH = "/Volumes/nyc_311/bronze/nyc_311_raw"

#BRONZE TABLE
BRONZE_TABLE = "nyc_311.bronze.raw_complaints"

print(f"Fetching 311 complaints created between {CREATED_DATE_FROM} and {CREATED_DATE_TO}")

Fetching 311 complaints created between 2025-01-01 and 2025-02-01


In [0]:
all_rows = []
offset = 0
page_size = 50000

while True:
    params = {
        "$limit": page_size,
        "$offset": offset,
        "$order": ":id",
        "$where": f"created_date >= '{CREATED_DATE_FROM}' AND created_date < '{CREATED_DATE_TO}'"
    }
    headers = {"X-App-Token": SOCRATA_APP_TOKEN}

    response = requests.get(SOCRATA_URL, params=params, headers=headers)
    response.raise_for_status()
    rows = response.json()

    if not rows:
        break

    all_rows.extend(rows)
    offset += len(rows)
    print(f"Fetched {len(all_rows)} rows so far...")
    time.sleep(0.5)

print(f"Total: {len(all_rows)} rows")

Fetched 50000 rows so far...
Fetched 100000 rows so far...
Fetched 150000 rows so far...
Fetched 200000 rows so far...
Fetched 250000 rows so far...
Fetched 300000 rows so far...
Fetched 348180 rows so far...
Total: 348180 rows


In [0]:
import os

# Create the raw folder if it doesn't exist
os.makedirs(f"{VOLUME_PATH}/raw", exist_ok=True)

# Save the fetched data
raw_path = f"{VOLUME_PATH}/raw/311_complaints_2025_01.json"
with open(raw_path, "w") as f:
    json.dump(all_rows, f)

print(f"✅ Saved {len(all_rows)} rows to {raw_path}")

✅ Saved 348180 rows to /Volumes/nyc_311/bronze/nyc_311_raw/raw/311_complaints_2025_01.json


In [0]:
raw_path = f"{VOLUME_PATH}/raw/311_complaints_2025_01.json"

df_raw = spark.read.json(raw_path)
print(f"Rows: {df_raw.count()}")
print(f"Columns: {len(df_raw.columns)}")
df_raw.printSchema()

Rows: 348180
Columns: 44
root
 |-- address_type: string (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- bridge_highway_direction: string (nullable = true)
 |-- bridge_highway_name: string (nullable = true)
 |-- bridge_highway_segment: string (nullable = true)
 |-- city: string (nullable = true)
 |-- closed_date: string (nullable = true)
 |-- community_board: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- cross_street_1: string (nullable = true)
 |-- cross_street_2: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- descriptor_2: string (nullable = true)
 |-- due_date: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- incident_zip: string (nullable

In [0]:
#Add metadata columns
df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit("socrata_api"))
    .withColumn("_source_url", F.lit(SOCRATA_URL))
    .withColumn("_date_range_from", F.lit(CREATED_DATE_FROM))
    .withColumn("_date_range_to", F.lit(CREATED_DATE_TO))
)

In [0]:
#Write to bronze delta table
df_bronze.write \
    .format("delta") \
        .mode("overwrite") \
            .option("overwriteSchema", "true") \
                .saveAsTable(BRONZE_TABLE)

print(f"✅ Bronze table created: {BRONZE_TABLE}")
print(f"Rows written: {spark.read.table(BRONZE_TABLE).count()}")
print(f"Columns: {len(spark.read.table(BRONZE_TABLE).columns)}")

✅ Bronze table created: nyc_311.bronze.raw_complaints
Rows written: 348180
Columns: 49


In [0]:
%sql
SELECT
    MIN(created_date) AS earliest,
    MAX(created_date) AS latest,
    COUNT(DISTINCT unique_key) AS unique_complaints,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT borough) as distinct_borough,
    COUNT(DISTINCT complaint_type) as distinct_types
FROM nyc_311.bronze.raw_complaints

earliest,latest,unique_complaints,total_rows,distinct_borough,distinct_types
2025-01-01T00:00:12.000,2025-01-31T23:59:48.000,348180,348180,6,168
